In [1]:
# Setup imports
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown as md
import time
import os
import serial
from pico import Pico

import serial.tools.list_ports
for port in serial.tools.list_ports.comports():
    print(port.device, port.description) # this should show the port where the Pi Pico

# sound feedback. use by calling beep(<number of beeps>)
beep = lambda x: os.system("echo -n '\a';sleep 0.2;" * x)

from datetime import datetime

import gc
import ctypes

/dev/ttyACM0 Board in FS mode - Board CDC


In [2]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [3]:
# Make graphs large enough to read easily
plt.clf()
plt.rcParams["figure.figsize"] = [10.0, 10.0*2/3]
matplotlib.rcParams["font.size"] = 20
None

# loads caen_daq code
display(md("### load the code"))
%run gui/wja_caen_tcal
tbegin = time.time()

# loads the drs4 corrections
f.load_drs_corrections()
print(list(f.tcal[0].__dict__.keys()))

### load the code

wja_caen_tcal.py starting at Thu Oct  8 12:13:37 2026


<Figure size 640x480 with 0 Axes>

['idrs', 'ichnl', 'tstamp', 'cellpeds', 'cellgain', 'celldt', 'wiggleshape', 'meanshape']


<Figure size 1000x666.667 with 0 Axes>

In [4]:
def acquire_and_plot():
    # acquire
    starttime = time.time()
    t0 = time.time()
    f.do_triggered_readout(nevents)
    t_acq = time.time() - t0
    print(list(f.trigev[0].__dict__.keys()))
    print(f"acquisition elapsed time {t_acq:.1f}s")
    print(f"rough rate = {nevents/t_acq:.1f}Hz")
    beep(1)
    t0 = time.time()
    f.correct_triggered_readout()
    t_cor = time.time() - t0
    print(f"corrections elapsed time {t_cor:.1f}s")
    beep(1)

    # plot
    t0 = time.time()
    if doplot:
        for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]:
            print(f"SiPM Channel {ich+1} of 16")
            for ev in range (min(10, nevents)):
                plt.plot(f.trigev[ev].drsu[ich])
            plt.show()
            print(f"----------------------")
    t_plot = time.time() - t0

    # Save to HDF5 file
    t0 = time.time()
    hdf5_fnam = f"{today}_{notes}_{xtaln}.hdf5"

    ns = SimpleNamespace()
    ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
    ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
    ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
    ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
    ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
    ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
    ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
    ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
    ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
    ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
    ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
    ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

    if "hf" in vars():
        hf.close()
        del hf

    hf = h5py.File(hdf5_fnam, "w")

    def cds(dsetname, comment):
        ds = hf.create_dataset(
            dsetname, 
            data=ns.__dict__[dsetname],
            shuffle=True,
            compression="gzip",
            compression_opts=1)
        ds.attrs["comment"] = comment

    cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
    cds("drs", "DRS samples with pedestal and gain corrections applied")
    cds("drsu", "DRS corrected samples, resampled to equal time intervals")
    cds("traw", "nominal uncorrected DRS sample times")
    cds("tcor", "DRS sample times, corrected via timing calibration")
    cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
    cds("drs_tstamp", "CAEN board time stamp [event]")
    cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
    cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
    cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
    cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
    cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
    hf.close()
    del hf
    t_h5 = time.time() - t0
    print(f"hdf5 elapsed time: {t_h5:.1f}s")
    t_total = time.time() - starttime
    print(f"total elapsed time {t_total:.1f}s")

    # --- timing record ---
    rec = dict(seq=len(acq_log), xtal=xtaln, nevents=nevents,
               t_acquire=t_acq, t_correct=t_cor, t_plot=t_plot,
               t_hdf5=t_h5, t_total=t_total,
               wall_start=starttime, hdf5_file=hdf5_fnam)
    acq_log.append(rec)
    
    write_header = not os.path.exists(acq_time_fnam)
    with open(acq_time_fnam, "a") as fh:
        if write_header:
            fh.write(",".join(rec.keys()) + "\n")
        fh.write(",".join(str(v) for v in rec.values()) + "\n")

    # --- release memory ---
    plt.close('all')
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)

In [5]:
pico = Pico("/dev/ttyACM0")   # replace with the actual port of the Pi Pico
print(repr(pico.exec("import Main as m"))) # loads Main.py from the Pico
pico.exec("import sys; del sys.modules['Main']; import Main as m") # this is a trick to reload the module in case it was already loaded, so that we get the latest version after uploading
print(repr(pico.exec("print(m.current_M_yaxis)"))) # Prints current M from y axis, just to test that things worked
print(pico.exec("m.enable_motor()")) # engages the motors. they should be hard to turn manually after this

✓ Uploaded Main.py → Main.py (23261 bytes)
<module 'Main' from 'Main.py'>
'import Main as m\r\n>>> '
0.0
'print(m.current_M_yaxis)\r\n0.0\r\n>>> '
m.enable_motor()
m.enable_motor()
>>> 


In [6]:
# REUPLOAD AND RELOAD
pico.upload("/home/emmo/proj/sipm/PiPico_MotorControls/Main.py")
pico.exec("import sys; del sys.modules['Main']; import Main as m")

✓ Uploaded /home/emmo/proj/sipm/PiPico_MotorControls/Main.py → Main.py (23261 bytes)


"import sys; del sys.modules['Main']; import Main as m\r\n>>> "

In [ ]:
pico.exec("m.calibrate('x')") # calibrate the x axis to setup its range of motion
# pico.exec("m.home('x')") # home the x axis

In [ ]:
pico.exec("m.calibrate('y')") # calibrate the y axis to setup its range of motion
# pico.exec("m.home('y')") # home the y axis

In [ ]:
# X coordinates from home
# pico.exec("m.move_distance_mm_x(20.55, True, True)") # col 1
x0 = 20.55
# pico.exec("m.move_distance_mm_x(19.05, True, True)") # col 2
x1 = 19.05
# pico.exec("m.move_distance_mm_x(17.47, True, True)") # col 3
x2 = 17.47
# pico.exec("m.move_distance_mm_x(15.97, True, True)") # col 4
x3 = 15.97
# pico.exec("m.move_distance_mm_x(14.39, True, True)") # col 5
x4 = 14.39
# pico.exec("m.move_distance_mm_x(12.89, True, True)") # col 6
x5 = 12.89
# pico.exec("m.move_distance_mm_x(11.31, True, True)") # col 7
x6 = 11.31
# pico.exec("m.move_distance_mm_x(9.81, True, True)") # col 8
x7 = 9.81
# pico.exec("m.move_distance_mm_x(8.23, True, True)") # col 9
x8 = 8.23
# pico.exec("m.move_distance_mm_x(6.73, True, True)") # col 10
x9 = 6.73
# pico.exec("m.move_distance_mm_x(13.61, True, True)") # xtal array center x
x_center = 13.61

# Y coordinates from home
# pico.exec("m.move_distance_mm_y(44.65, True, True)") # row 1
y0 = 44.65
# pico.exec("m.move_distance_mm_y(43.15, True, True)") # row 2
y1 = 43.15
# pico.exec("m.move_distance_mm_y(41.57, True, True)") # row 3
y2 = 41.57
# pico.exec("m.move_distance_mm_y(40.07, True, True)") # row 4
y3 = 40.07
# pico.exec("m.move_distance_mm_y(38.49, True, True)") # row 5
y4 = 38.49
# pico.exec("m.move_distance_mm_y(36.99, True, True)") # row 6
y5 = 36.99
# pico.exec("m.move_distance_mm_y(35.41, True, True)") # row 7
y6 = 35.41
# pico.exec("m.move_distance_mm_y(33.91, True, True)") # row 8
y7 = 33.91
# pico.exec("m.move_distance_mm_y(32.33, True, True)") # row 9
y8 = 32.33
# pico.exec("m.move_distance_mm_y(30.83, True, True)") # row 10
y9 = 30.83
# pico.exec("m.move_distance_mm_y(37.74, True, True)") # xtal arraycenter y
y_center = 37.74

# motion is to start at crystal 100, acquiring the row, moving up from the end of the row, acquire the row, and continue to snake up to crystal 10.
# 100 --> 91
# 81 --> 90
# 80 --> 71
# 61 --> 70
# 60 --> 51
# 41 --> 50
# 40 --> 31
# 21 --> 30
# 20 --> 11
# 1 --> 10


In [ ]:
nevents = 100
today = datetime.now().strftime("%Y%m%d")
notes = f"hfv2_TAC3x3x3_TAC10x10array_47p5V_Ref450mV_6db_Sum200mV_6db_timing_{nevents}events_CAEN"
acq_log = [] # one record per call, in visiting order
acq_time_fnam = f"{today}_{notes}_acq-times.csv"
doplot = True # Set to False to skip plotting

In [ ]:
# DEBUGGING ONLY
# print(f"{y8-y9}")
# pico.exec("m.home('x')") # home the x axis
# pico.exec("m.home('y')") # home the y axis
# pico.exec(f"m.move_distance_mm_x({x_center}, True, True)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y_center}, True, True)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_x({x9-1.66}, True, True)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y9-1.66}, True, True)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_x(9, True, True)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y(9, True, True)") # (distance in mm, True for fast speed, True for forward)

In [ ]:
xtaln = "xtal100"

# moving to xtal 100 center
pico.exec(f"m.move_distance_mm_x({x9}, True, True)") # (distance in mm, True for fast speed, True for forward)
pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)


In [ ]:
# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal099"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, True)") # (distance in mm, True for fast speed, True for forward)
#pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal098"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, True)") # (distance in mm, True for fast speed, True for forward)
#pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal097"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, True)") # (distance in mm, True for fast speed, True for forward)
#pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal096"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, True)") # (distance in mm, True for fast speed, True for forward)
#pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal095"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, True)") # (distance in mm, True for fast speed, True for forward)
#pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal094"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, True)") # (distance in mm, True for fast speed, True for forward)
#pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal093"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, True)") # (distance in mm, True for fast speed, True for forward)
#pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal092"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, True)") # (distance in mm, True for fast speed, True for forward)
#pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal091"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)") # (distance in mm, True for fast speed, True for forward)
#pico.exec(f"m.move_distance_mm_y({y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal081"

# moving to xtal center
#pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)") # (distance in mm, True for fast speed, True for forward)
pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal082"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, False)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal083"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, False)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal084"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, False)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal085"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, False)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal086"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, False)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal087"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, False)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal088"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, False)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal089"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, False)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal090"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, False)") # (distance in mm, True for fast speed, True for forward)
# pico.exec(f"m.move_distance_mm_y({y8-y9}, True, True)") # (distance in mm, True for fast speed, True for forward)

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal080"

# moving to xtal center
# pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, False)")
pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal079"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal078"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal077"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal076"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal075"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal074"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal073"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal072"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal071"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y7-y8}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal061"

# moving to xtal center
# pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)")
pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal062"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal063"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal064"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal065"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal066"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal067"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal068"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal069"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal070"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y6-y7}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal060"

# moving to xtal center
# pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, False)")
pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal059"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal058"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal057"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal056"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal055"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal054"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal053"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal052"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal051"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y5-y6}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal041"

# moving to xtal center
# pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)")
pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal042"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal043"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal044"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal045"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal046"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal047"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal048"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal049"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal050"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y4-y5}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal040"

# moving to xtal center
# pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, False)")
pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal039"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal038"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal037"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal036"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal035"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal034"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal033"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal032"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal031"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y3-y4}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal021"

# moving to xtal center
# pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)")
pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal022"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal023"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal024"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal025"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal026"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal027"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal028"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal029"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal030"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y2-y3}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal020"

# moving to xtal center
# pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, False)")
pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal019"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal018"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal017"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal016"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal015"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal014"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal013"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal012"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal011"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)")
# pico.exec(f"m.move_distance_mm_y({y1-y2}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal001"

# moving to xtal center
# pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, True)")
pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal002"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x0-x1}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal003"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x1-x2}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal004"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x2-x3}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal005"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x3-x4}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal006"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x4-x5}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal007"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x5-x6}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal008"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x6-x7}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal009"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x7-x8}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
xtaln = "xtal010"

# moving to xtal center
pico.exec(f"m.move_distance_mm_x(({x8-x9}), True, False)")
# pico.exec(f"m.move_distance_mm_y({y0-y1}, True, True)")

# acquire
acquire_and_plot()

In [ ]:
# beep(2)
time.sleep(120)

In [ ]:
# YOU NEED TO MANUALLY SAVE THE NOTEBOOK FIRST!
import subprocess
from datetime import datetime

notebook = "sipm_10x10_xtal_array_daq_v0.ipynb"
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output = f"{notes}_{timestamp}.html"

subprocess.run(
    ["jupyter", "nbconvert", "--to", "html", notebook, "--output", output],
    check=True
)
print(f"Exported: {output}")
# YOU NEED TO MANUALLY SAVE THE NOTEBOOK FIRST!

In [ ]:
# run export_to_html --timestamp # doesnt work in vscode. need to be in a browser.